In [ ]:
import unicodedata
from nltk.sem.logic import Expression
import re

def fol_preprocess(expression):
    replacements = {
        '∀': ' FORALL ', '∃': ' EXISTS ', '¬': 'NOT ', '∧': 'AND',
        '⊕': 'XOR', '∨': 'OR', '→': 'THEN', '↔': 'IFF',
    }
    for symbol, replacement in replacements.items():
        expression = expression.replace(symbol, replacement)
    return expression

def fol_postprocess(text):
    inv_replacements = {
        'FORALL': '∀', 'EXISTS': '∃', 'NOT': '¬', 'AND': '∧',
        'XOR': '⊕', 'OR': '∨', 'THEN': '→', 'IFF': '↔',
    }
    for replacement, symbol in inv_replacements.items():
        text = text.replace(replacement, symbol)
    return text

def nl_preprocess(sentence):
    return unicodedata.normalize('NFKC', sentence.lower())

def to_nltk(expression):
    replacements = {
        '∀': ' all ', '∃': ' exists ', '¬': '-', '∧': '&', '⊕': '!=',
        '∨': '|', '→': '->', '↔': '<->', 'FORALL': ' all ', 'EXISTS': ' exists ',
        'NOT': ' - ', 'AND': '&', 'XOR': '!=', 'OR': '|', 'THEN': '->', 'IFF': '<->',
    }
    for symbol, nltk in replacements.items():
        expression = re.sub(re.escape(symbol), nltk, expression)
    return expression

def check_grammar(expression):
    nltk_expression = to_nltk(expression)
    try:
        parsed_expr = Expression.fromstring(nltk_expression)
        return len(parsed_expr.free()) == 0
    except Exception:
        return False

In [ ]:
import torch
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from tqdm import tqdm

model_path = ''
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSeq2SeqLM.from_pretrained(model_path).to("cuda" if torch.cuda.is_available() else "cpu")
model.eval()

def generate_fol(nl_sentence):
    input_text = "translate English to First-order Logic: " + nl_sentence.lower()
    inputs = tokenizer(input_text, return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs, 
            max_length=1024, 
            num_beams=4, 
            early_stopping=True
        )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

all_preds = ["human", "tall", "smart", "kind", "brave", "calm", "wise", "bold", "fast", "strong"]
all_names = ["Rebecca", "John", "Michael", "Brian", "Lucas", "Sarah", "Robert", "Alice", "David", "Laura"]

test_suite = []

for start_idx in range(len(all_names)):
    for i in range(1, 11):
        indices = [(start_idx + j) % len(all_names) for j in range(i)]
        current_names = [all_names[idx] for idx in indices]
        test_suite.append({"depth": i, "type": "CONJ", "nl": " and ".join([f"{n} is a human" for n in current_names]) + "."})
        test_suite.append({"depth": i, "type": "DISJ", "nl": " or ".join([f"{n} is a human" for n in current_names]) + "."})

for subject in all_names:
    for i in range(1, 11):
        current_attrs = all_preds[:i]
        test_suite.append({"depth": i, "type": "VERT_CONJ", "nl": " and ".join([f"{subject} is {a}" for a in current_attrs]) + "."})
        test_suite.append({"depth": i, "type": "VERT_DISJ", "nl": " or ".join([f"{subject} is {a}" for a in current_attrs]) + "."})

def diagnostic_eval(suite, all_possible_preds, all_possible_names):
    results = []
    print(f"Running {len(suite)} strict inferences...")
    
    for case in tqdm(suite, desc="Evaluating"):
        prediction = generate_fol(case["nl"])
        pred_lower = prediction.lower()
        nl_lower = case["nl"].lower()
        
        omitted_count = 0
        redundancy_count = 0
        is_success = True
        
        is_well_formed = check_grammar(prediction)
        if not is_well_formed:
            is_success = False
            
        if "VERT" in case["type"]:
            for p in all_possible_preds:
                expected_c = 1 if p.lower() in nl_lower else 0
                actual_c = pred_lower.count(p.lower())
                
                if actual_c < expected_c:
                    omitted_count += (expected_c - actual_c)
                    is_success = False
                elif actual_c > expected_c:
                    redundancy_count += (actual_c - expected_c)
                    is_success = False
                    
        else:
            for n in all_possible_names:
                expected_c = 1 if n.lower() in nl_lower else 0
                actual_c = pred_lower.count(n.lower())
                
                if actual_c < expected_c:
                    omitted_count += (expected_c - actual_c)
                    is_success = False
                elif actual_c > expected_c:
                    redundancy_count += (actual_c - expected_c)
                    is_success = False
            
            expected_human_c = case["depth"]
            actual_human_c = pred_lower.count("human")
            if actual_human_c != expected_human_c:
                is_success = False
                if actual_human_c < expected_human_c:
                    omitted_count += (expected_human_c - actual_human_c)
                else:
                    redundancy_count += (actual_human_c - expected_human_c)

        results.append({
            "NL_Input": case["nl"],
            "FOL_Prediction": prediction,
            "Type": case["type"],
            "Depth": case["depth"],
            "Success": "PASS" if is_success else "FAIL",
            "Well_Formed": is_well_formed,
            "Omitted_Count": omitted_count,
            "Redundancy_Count": redundancy_count
        })
    return pd.DataFrame(results)

report_df = diagnostic_eval(test_suite, all_possible_preds=all_preds, all_possible_names=all_names)
report_df.to_excel("experiment_3_results.xlsx", index=False)

print("\n--- Exact Match Success Rate (%) by Depth and Type ---")
success_summary = report_df.groupby(["Depth", "Type"])["Success"].apply(lambda x: (x == "PASS").mean() * 100).unstack()
print(success_summary[['CONJ', 'DISJ', 'VERT_CONJ', 'VERT_DISJ']].round(1))

print("\n--- Syntactic Well-Formedness (%) by Depth ---")
grammar_summary = report_df.groupby("Depth")["Well_Formed"].mean() * 100
print(grammar_summary.round(1))

print("\n--- Average Omissions per Expression by Depth and Type ---")
omission_summary = report_df.groupby(["Depth", "Type"])["Omitted_Count"].mean().unstack()
print(omission_summary[['CONJ', 'DISJ', 'VERT_CONJ', 'VERT_DISJ']].round(2))

print("\n--- Average Redundancies per Expression by Depth and Type ---")
redundancy_summary = report_df.groupby(["Depth", "Type"])["Redundancy_Count"].mean().unstack()
print(redundancy_summary[['CONJ', 'DISJ', 'VERT_CONJ', 'VERT_DISJ']].round(2))

In [ ]:
import pandas as pd

def analyze_experiment_results(file_path):
    print(f"Loading data from: {file_path}")
    
    try:
        df = pd.read_excel(file_path)
    except FileNotFoundError:
        print(f"Error: The file '{file_path}' was not found.")
        return

    df['Is_Success'] = df['Success'] == "PASS"

    print("\n--- Exact Match Success Rate (%) by Depth and Type ---")
    success_summary = df.groupby(["Depth", "Type"])["Is_Success"].mean() * 100
    success_summary = success_summary.unstack()
    print(success_summary[['CONJ', 'DISJ', 'VERT_CONJ', 'VERT_DISJ']].round(1))

    print("\n--- Syntactic Well-Formedness (%) by Depth ---")
    grammar_summary = df.groupby("Depth")["Well_Formed"].mean() * 100
    print(grammar_summary.round(1))

    print("\n--- Average Omissions per Expression by Depth and Type ---")
    omission_summary = df.groupby(["Depth", "Type"])["Omitted_Count"].mean().unstack()
    print(omission_summary[['CONJ', 'DISJ', 'VERT_CONJ', 'VERT_DISJ']].round(2))

    print("\n--- Average Redundancies per Expression by Depth and Type ---")
    redundancy_summary = df.groupby(["Depth", "Type"])["Redundancy_Count"].mean().unstack()
    print(redundancy_summary[['CONJ', 'DISJ', 'VERT_CONJ', 'VERT_DISJ']].round(2))

if __name__ == "__main__":
    EXCEL_FILE = "experiment_3_results_small.xlsx"
    analyze_experiment_results(EXCEL_FILE)